In [ ]:
import requests
import pandas as pd
import time

API_KEY = '696fbbad611cc1.68987853'

In [ ]:
ticker = 'AAPL.US'
url = f'https://eodhd.com/api/eod/{ticker}?api_token={API_KEY}&fmt=json'

response = requests.get(url)
data = response.json()
df = pd.DataFrame(data)
print(df.head())

         date     open     high      low    close  adjusted_close     volume
0  1980-12-12  28.7392  28.8736  28.7392  28.7392          0.0983  469033600
1  1980-12-15  27.3728  27.3728  27.2608  27.2608          0.0932  175884800
2  1980-12-16  25.3792  25.3792  25.2448  25.2448          0.0863  105728000
3  1980-12-17  25.8720  26.0064  25.8720  25.8720          0.0885   86441600
4  1980-12-18  26.6336  26.7456  26.6336  26.6336          0.0911   73449600


MANIPULATE THE DATE

In [ ]:
if 'date' in df.columns:
  df['date'] = pd.to_datetime(df['date'])
  df.set_index('date', inplace=True)
  df = df.sort_index()

latest_day = df.index[-1]

ONE YEAR CUMULATIVE RETURN CALCULATION

In [ ]:
date_1_year_ago = latest_day - pd.DateOffset(months = 12)
date_1_month_ago = latest_day - pd.DateOffset(months = 1)

p_last_month = df['adjusted_close'].asof(date_1_month_ago)
p_last_year = df['adjusted_close'].asof(date_1_year_ago)

MOM_12 = ((p_last_month/p_last_year) - 1) * 100

print(f"12-month momentum: {MOM_12:.2f} %")

12-month momentum: 0.70 %


SIX MONTHS CUMULATIVE RETURN CALCULATION

In [ ]:
date_6_months_ago = latest_day - pd.DateOffset(months = 6)
date_1_month_ago = latest_day - pd.DateOffset(months = 1)

p_last_month = df['adjusted_close'].asof(date_1_month_ago)
p_last_6_months = df['adjusted_close'].asof(date_6_months_ago)

MOM_6 = ((p_last_month/p_last_6_months) - 1) * 100

print(f"6-month momentum: {MOM_6:.2f} %")

6-month momentum: 9.26 %


THREE MONTHS CUMULATIVE RETURN CALCULATION


In [ ]:
date_3_months_ago = latest_day - pd.DateOffset(months = 3)
date_1_month_ago = latest_day - pd.DateOffset(months = 1)

p_last_month = df['adjusted_close'].asof(date_1_month_ago)
p_last_3_months = df['adjusted_close'].asof(date_3_months_ago)

MOM_3 = ((p_last_month/p_last_3_months) - 1) * 100

print(f"3-month momentum: {MOM_3:.2f} %")

3-month momentum: -7.34 %


RSI CALCULATION

In [ ]:
from pandas.core.window.rolling import Window
delta = df['adjusted_close'].diff()

gain = (delta.where(delta > 0,0))
loss = (-delta.where(delta < 0,0))

avg_gain = gain.rolling(window = 14).mean()
avg_loss = loss.rolling(window = 14).mean()

RS = avg_gain/avg_loss
RSI = 100 - 100/(1+RS)
print(f"The stock's RSI is: {RSI.iloc[-1]}")

The stock's RSI is: 54.24687283387704


DISTANCE BETWEEN PRICE AND MA200 CALCULATION


In [ ]:
ma200 = df['adjusted_close'].rolling(window = 200).mean()
real_time_price = df['adjusted_close']
dist_ma200 = ((real_time_price/ma200) - 1)*100

print(f"The stock's distance from MA200 is: {dist_ma200.iloc[-1]:.2f} %")

The stock's distance from MA200 is: 9.90 %


VOLATILITY METRIC CALCULATION

In [ ]:
import numpy as np

returns = df['adjusted_close'].pct_change()
volatility = returns.rolling(window=20).std() * np.sqrt(252)

current_vol = volatility.iloc[-1]
print(f"Volatility (20d Annualized): {current_vol:.2%}")

Volatility (20d Annualized): 32.47%
